### Vectorless RAG

In [2]:
import os, json, time
from dotenv import load_dotenv
load_dotenv()

pageindex_api_key = os.getenv("PageIndex_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

print("PageIndex key loaded: present" if pageindex_api_key else "Missing")
print("groq_api_key: present" if groq_api_key else "Missing")

PageIndex key loaded: present
groq_api_key: present


In [3]:
from pageindex import PageIndexClient
from langchain_groq import ChatGroq

pi_client = PageIndexClient(api_key=pageindex_api_key)
groq_client = ChatGroq(model="llama-3.3-70b-versatile")

print("PageIndex client is ready")
print("groq client is ready")

d:\LangChain_Learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PageIndex client is ready
groq client is ready


---
## 🌲 Section 2: Upload & Index a PDF

**What happens here:**
1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical **tree index** (like a smart Table of Contents)
4. Returns a `doc_id` for all future operations

**Why NO chunking?**  
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.

In [ ]:
PDF_PATH = "./langchain_interview_guide.pdf"

print(f"Uploading:{PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"Uploaded")
print(f"Document id: {doc_id}")

Uploading:./langchain_interview_guide.pdf
Uploaded
Document id: pi-cmrqtinat00zn01qurf89m0ie


In [6]:
print("pi-cmrqtinat00zn01qurf89m0ie")

pi-cmrqtinat00zn01qurf89m0ie


In [7]:
# Page Index builds the tree asynchronously.
# for a 50 page pdf this typically takes 30-90 seconds.

print("Building Tree Index")
print("This runs once per document - the index is cached for reuse")

while True:
    status_reuslt = pi_client.get_document(doc_id)
    status = status_reuslt.get("status")
    print(f" Status: {status}")

    if status == "completed":
        print("\n Tree Index ready")
        break
    elif status == "failed":
        print("\n Processing failed. Check the pdf format")
        break

    time.sleep(5)

Building Tree Index
This runs once per document - the index is cached for reuse
 Status: completed

 Tree Index ready


---
## 🔍 Section 3: Inspect the Tree Structure

**What the tree looks like:**

```
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**


In [8]:
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result",[])

print(f"Top level sections: {len(pageindex_tree)}")
print("\n Raw tree(first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top level sections: 1

 Raw tree(first node):
{
  "title": "LangChain Interview Guide (Practical, Layman-Terms Edition)",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# LangChain Interview Guide (Practical, Layman-Terms Edition)\n\nWritten the way a 6-7 YOE Agentic AI developer would actually get quizzed \u2014 concept first in plain English, then the code an interviewer expects you to write on a whiteboard or shared editor.\n",
  "text": "# LangChain Interview Guide (Practical, Layman-Terms Edition)\n\nWritten the way a 6-7 YOE Agentic AI developer would actually get quizzed \u2014 concept first in plain English, then the code an interviewer expects you to write on a whiteboard or shared editor.\n",
  "nodes": [
    {
      "title": "SECTION 1: Fundamentals (they check if you actually understand the \u201cwhy\u201d)",
      "node_id": "0001",
      "page_index": 1,
      "summary": "## SECTION 1: Fundamentals (they check if you actually understand the \u201cwhy\u201d)\

In [9]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] LangChain Interview Guide (Practical, Layman-Terms Edition)  (p.1)
  └─ [0001] SECTION 1: Fundamentals (they check if you actually understand the “why”)  (p.1)
  └─ [0002] SECTION 2: Practical Coding (hands-on, they want to see you type)  (p.2)
    └─ [0003] Q4. Build a simple LCEL chain and explain what the | pipe actually does.  (p.2)
    └─ [0004] Q5. How does memory work in a chatbot built with LangChain?  (p.3)
    └─ [0005] Q6. Write a custom Tool and wire it into an agent.  (p.3)
  └─ [0006] SECTION 3: RAG & Retrieval (almost guaranteed in any agentic AI interview)  (p.4)
    └─ [0007] Q7. Explain RAG in one paragraph, no jargon.  (p.4)
      └─ [0008] 1. Load  (p.4)
      └─ [0009] 2. Split into chunks (overlap keeps context from being cut mid-sentence)  (p.4)
      └─ [0010] 3. Embed + store  (p.5)
      └─ [0011] 4. Retrieve + 5. Answer  (p.5)
      └─ [0012] Score-based filtering to drop weak matches before they reach the prompt  (p.5)
    

In [10]:
# Count the total number of nodes
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f" Total nodes in a tree: {total}")
print("Each nodes = one retrieval section of the document")

 Total nodes in a tree: 24
Each nodes = one retrieval section of the document


---
## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

### Vector RAG retrieval:
```
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```
*Problem: finds what's similar, not what's relevant*

### PageIndex retrieval:
```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**


In [23]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "llama-3.3-70b-versatile") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.bind(
        model=model,
        # messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    response = response.invoke(prompt)
    
    return json.loads(response.content)

In [24]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is langchain?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is langchain?

🧠 LLM Reasoning:
To identify the node IDs most likely to contain the answer to the query 'What is langchain?', we should look for sections and subsections that provide introductory or explanatory information about LangChain. The document tree structure suggests that sections with titles like 'LangChain Interview Guide', 'Fundamentals', and 'Practical Coding' might be relevant. However, since the query is about the definition or explanation of LangChain, we should focus on the top-level introduction and any subsections that might provide an overview or basic understanding of LangChain. The node with ID '0000' seems to be the introduction to the LangChain interview guide, and its children might provide more specific insights into what LangChain is, especially those related to fundamentals and practical coding aspects.

🎯 Selected Node IDs: ['0000', '0001', '0004']
